# Histopathology — MIL aggregator over frozen embeddings (Google Colab)

Colab port of [`notebooks/kaggle/02_pathology_mil_training.ipynb`](../kaggle/02_pathology_mil_training.ipynb).
This is the easiest of the four to port — the Kaggle version already loads
**PatchCamelyon via the Hugging Face `datasets` library**, not a Kaggle-specific
dataset mount, so the data-loading code is identical here. Only the GPU check
and Drive-based checkpoint storage are Colab-specific.

See the Kaggle notebook for the full reasoning (why patches not full WSIs, why
the embedding backbone stays frozen, the synthetic-bags caveat, etc.).

## Colab setup
1. **Runtime → Change runtime type → T4 GPU.**
2. Mount Drive (cell 2) so the trained aggregator checkpoint survives a
   disconnect — the embedding extraction step (cell 6) is the slow part, so
   losing it to a disconnect is the most annoying failure mode to avoid.

In [ ]:
# Cell 1 — install packages.
!pip install -q datasets transformers accelerate

In [ ]:
# Cell 2 — mount Drive for persistent checkpoint + cached-embedding storage.
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_WORKDIR = "/content/drive/MyDrive/aegis-dx/pathology-mil"
os.makedirs(DRIVE_WORKDIR, exist_ok=True)
print("Persistent working dir:", DRIVE_WORKDIR)

In [ ]:
# Cell 3 — imports and reproducibility.
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected - check Runtime > Change runtime type > T4 GPU.")

In [ ]:
# Cell 4 — load PatchCamelyon splits (identical to the Kaggle notebook).
TRAIN_PATCH_LIMIT = 20_000
VAL_PATCH_LIMIT = 4_000
TEST_PATCH_LIMIT = 4_000

raw_train = load_dataset("dpdl-benchmark/patch_camelyon", split=f"train[:{TRAIN_PATCH_LIMIT}]")
raw_val = load_dataset("dpdl-benchmark/patch_camelyon", split=f"validation[:{VAL_PATCH_LIMIT}]")
raw_test = load_dataset("dpdl-benchmark/patch_camelyon", split=f"test[:{TEST_PATCH_LIMIT}]")
print(raw_train, raw_val, raw_test)

In [ ]:
# Cell 5 — frozen embedding backbone (same caveat as Kaggle: swap for
# MedGemma's vision tower once you've confirmed its extraction path).
EMBED_MODEL_NAME = "google/vit-base-patch16-224-in21k"

image_processor = AutoImageProcessor.from_pretrained(EMBED_MODEL_NAME)
embed_model = AutoModel.from_pretrained(EMBED_MODEL_NAME).to(DEVICE).eval()
for parameter in embed_model.parameters():
    parameter.requires_grad_(False)

EMBED_DIM = embed_model.config.hidden_size
print("Embedding dimension:", EMBED_DIM)

## Embed once, cache to Drive

Embedding extraction is the slow step. We cache the resulting tensors to Drive
so a disconnect during training doesn't force you to redo it — re-running this
notebook later loads the cache instead of re-embedding.

In [ ]:
# Cell 6 — embed every patch once, cache to Drive.
@torch.no_grad()
def embed_split(raw_split, batch_size: int = 64) -> tuple[torch.Tensor, torch.Tensor]:
    embeddings, labels = [], []
    for start in range(0, len(raw_split), batch_size):
        batch = raw_split[start : start + batch_size]
        pixel_values = image_processor(images=batch["image"], return_tensors="pt").pixel_values.to(DEVICE)
        pooled = embed_model(pixel_values=pixel_values).last_hidden_state[:, 0, :]
        embeddings.append(pooled.cpu())
        labels.extend(batch["label"])
    return torch.cat(embeddings), torch.tensor(labels, dtype=torch.float32)


def embed_or_load_cached(raw_split, cache_name: str):
    cache_path = os.path.join(DRIVE_WORKDIR, cache_name)
    if os.path.exists(cache_path):
        cached = torch.load(cache_path)
        return cached["embeddings"], cached["labels"]
    embeddings, labels = embed_split(raw_split)
    torch.save({"embeddings": embeddings, "labels": labels}, cache_path)
    return embeddings, labels


train_embeddings, train_labels = embed_or_load_cached(raw_train, "train_embeddings.pt")
val_embeddings, val_labels = embed_or_load_cached(raw_val, "val_embeddings.pt")
test_embeddings, test_labels = embed_or_load_cached(raw_test, "test_embeddings.pt")
print(train_embeddings.shape, val_embeddings.shape, test_embeddings.shape)

In [ ]:
# Cell 7 — group embeddings into synthetic bags (same caveat as Kaggle: a
# real deployment groups by actual slide ID, not random grouping).
BAG_SIZE = 20


def make_bags(embeddings: torch.Tensor, labels: torch.Tensor, bag_size: int, seed: int):
    generator = torch.Generator().manual_seed(seed)
    permutation = torch.randperm(len(embeddings), generator=generator)
    bags, bag_labels = [], []
    for start in range(0, len(permutation) - bag_size + 1, bag_size):
        indices = permutation[start : start + bag_size]
        bags.append(embeddings[indices])
        bag_labels.append(float(labels[indices].max()))
    return bags, torch.tensor(bag_labels, dtype=torch.float32)


train_bags, train_bag_labels = make_bags(train_embeddings, train_labels, BAG_SIZE, seed=SEED)
val_bags, val_bag_labels = make_bags(val_embeddings, val_labels, BAG_SIZE, seed=SEED + 1)
test_bags, test_bag_labels = make_bags(test_embeddings, test_labels, BAG_SIZE, seed=SEED + 2)
print(f"bags: train={len(train_bags)} val={len(val_bags)} test={len(test_bags)}")


class BagDataset(Dataset):
    def __init__(self, bags: list[torch.Tensor], bag_labels: torch.Tensor):
        self.bags = bags
        self.bag_labels = bag_labels

    def __len__(self) -> int:
        return len(self.bags)

    def __getitem__(self, index: int):
        return self.bags[index], self.bag_labels[index]


train_loader = DataLoader(BagDataset(train_bags, train_bag_labels), batch_size=1, shuffle=True)
val_loader = DataLoader(BagDataset(val_bags, val_bag_labels), batch_size=1, shuffle=False)
test_loader = DataLoader(BagDataset(test_bags, test_bag_labels), batch_size=1, shuffle=False)

In [ ]:
# Cell 8 — attention-based MIL aggregator (identical to Kaggle).
class AttentionMIL(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, patch_embeddings: torch.Tensor):
        attention_logits = self.attention(patch_embeddings).squeeze(-1)
        attention_weights = F.softmax(attention_logits, dim=0)
        bag_embedding = (attention_weights.unsqueeze(-1) * patch_embeddings).sum(dim=0)
        bag_logit = self.classifier(bag_embedding)
        return bag_logit.squeeze(-1), attention_weights


mil_model = AttentionMIL(embed_dim=EMBED_DIM).to(DEVICE)
print(sum(p.numel() for p in mil_model.parameters()), "parameters")

In [ ]:
# Cell 9 — train the MIL aggregator, checkpointing best-so-far to Drive.
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(mil_model.parameters(), lr=1e-4, weight_decay=1e-5)
CHECKPOINT_PATH = os.path.join(DRIVE_WORKDIR, "pathology_mil_aggregator_best.pt")


@torch.no_grad()
def evaluate_mil(loader: DataLoader) -> float:
    mil_model.eval()
    all_true, all_score = [], []
    for patch_embeddings, bag_label in loader:
        patch_embeddings = patch_embeddings.squeeze(0).to(DEVICE)
        logit, _ = mil_model(patch_embeddings)
        all_true.append(bag_label.item())
        all_score.append(torch.sigmoid(logit).item())
    return roc_auc_score(all_true, all_score)


NUM_EPOCHS = 15
best_val_auroc = 0.0

for epoch in range(NUM_EPOCHS):
    mil_model.train()
    running_loss = 0.0
    for patch_embeddings, bag_label in train_loader:
        patch_embeddings = patch_embeddings.squeeze(0).to(DEVICE)
        bag_label = bag_label.to(DEVICE)
        optimizer.zero_grad()
        logit, _ = mil_model(patch_embeddings)
        loss = criterion(logit.unsqueeze(0), bag_label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    val_auroc = evaluate_mil(val_loader)
    print(f"epoch {epoch+1:02d}  train_loss={running_loss/len(train_loader):.4f}  val_auroc={val_auroc:.4f}")

    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        torch.save(mil_model.state_dict(), CHECKPOINT_PATH)

mil_model.load_state_dict(torch.load(CHECKPOINT_PATH))
print("Best val AUROC:", best_val_auroc)

In [ ]:
# Cell 10 — held-out test evaluation + save metadata sidecar.
test_auroc = evaluate_mil(test_loader)
print("Test bag-level AUROC:", test_auroc)

metadata = {
    "model": "AttentionMIL",
    "embedding_backbone": EMBED_MODEL_NAME,
    "embed_dim": EMBED_DIM,
    "bag_size": BAG_SIZE,
    "test_bag_auroc": test_auroc,
    "trained_on": "PatchCamelyon (dpdl-benchmark/patch_camelyon), synthetic bags",
    "trained_on_platform": "Google Colab",
    "caveat": "Real deployment must group patches by actual slide ID, not random bags.",
}
with open(os.path.join(DRIVE_WORKDIR, "pathology_mil_aggregator_best.json"), "w") as handle:
    json.dump(metadata, handle, indent=2)
print(json.dumps(metadata, indent=2))

## Next steps

Same as the Kaggle version: swap synthetic bags for real slide-grouped bags
before trusting this AUROC, confirm the embedding backbone choice, then
subgroup breakdown + calibration + registry entry. See
`notebooks/kaggle/02_pathology_mil_training.ipynb`'s final cell.